## **Aim**
To implement a program that performs WHOIS information collection for a domain as an OSINT investigation technique.

## **Algorithm**
**Step 1:** Import `socket`, `json`, `datetime`, and `subprocess` libraries.

**Step 2:** Create a simulated WHOIS database for domains since we can't do real WHOIS queries in this environment.

**Step 3:** Define a function `query_whois(domain)` that returns WHOIS information including:
   - Registrar name
   - Registration date
   - Expiration date
   - Name servers
   - Registrant info (if not private)
   - Domain status
   - DNSSEC status

**Step 4:** Parse the WHOIS response and extract key fields.

**Step 5:** Check for privacy protection services.

**Step 6:** Generate a structured WHOIS report.

In [1]:
import socket
import json
from datetime import datetime, timedelta

SIMULATED_WHOIS = {
    "example.com": {
        "registrar": "ICANN (Reserved)",
        "registrar_iana_id": 9999,
        "registrar_url": "https://www.iana.org/domains/example",
        "creation_date": "1995-08-14 04:00:00",
        "expiration_date": "2026-08-13 04:00:00",
        "updated_date": "2024-08-14 07:01:00",
        "name_servers": ["a.iana-servers.net", "b.iana-servers.net"],
        "registrant_name": "ICANN",
        "registrant_org": "Internet Corporation for Assigned Names and Numbers",
        "registrant_email": "abuse@iana.org",
        "registrant_phone": "+1.3103015800",
        "registrant_address": "12025 Waterfront Drive, Suite 300, Los Angeles, CA 90094, US",
        "status": ["clientDeleteProhibited", "clientTransferProhibited", "clientUpdateProhibited"],
        "dnssec": "Signed",
        "dnssec_algorithm": "8 (RSASHA256)",
        "privacy_protected": False
    },
    "paypal-security-update.tk": {
        "registrar": "Freenom",
        "registrar_iana_id": 1601,
        "registrar_url": "https://www.freenom.com",
        "creation_date": "2026-08-15 12:30:00",
        "expiration_date": "2027-08-15 12:30:00",
        "updated_date": "2026-08-15 12:30:00",
        "name_servers": ["ns1.freenom.com", "ns2.freenom.com"],
        "registrant_name": "REDACTED FOR PRIVACY",
        "registrant_org": "Privacy Protection Service",
        "registrant_email": "redacted@freenom.com",
        "registrant_phone": "+1.0000000000",
        "registrant_address": "REDACTED, REDACTED, REDACTED, REDACTED, US",
        "status": ["clientTransferProhibited", "serverTransferProhibited"],
        "dnssec": "Unsigned",
        "privacy_protected": True,
        "privacy_service": "Freenom Privacy Service"
    },
    "microsoft-login-verify.ml": {
        "registrar": "Freenom",
        "registrar_iana_id": 1601,
        "registrar_url": "https://www.freenom.com",
        "creation_date": "2026-08-10 09:15:00",
        "expiration_date": "2027-08-10 09:15:00",
        "updated_date": "2026-08-10 09:15:00",
        "name_servers": ["ns1.freenom.com", "ns2.freenom.com"],
        "registrant_name": "REDACTED FOR PRIVACY",
        "registrant_org": "Privacy Protection Service",
        "registrant_email": "redacted@freenom.com",
        "registrant_phone": "+1.0000000000",
        "registrant_address": "REDACTED, REDACTED, REDACTED, REDACTED, US",
        "status": ["clientTransferProhibited", "serverTransferProhibited"],
        "dnssec": "Unsigned",
        "privacy_protected": True,
        "privacy_service": "Freenom Privacy Service"
    },
    "github.com": {
        "registrar": "MarkMonitor Inc.",
        "registrar_iana_id": 292,
        "registrar_url": "https://www.markmonitor.com",
        "creation_date": "2007-10-09 18:20:50",
        "expiration_date": "2027-10-09 18:20:50",
        "updated_date": "2024-08-14 14:22:00",
        "name_servers": ["ns1.github.com", "ns2.github.com", "ns3.github.com", "ns4.github.com"],
        "registrant_name": "GitHub, Inc.",
        "registrant_org": "GitHub, Inc.",
        "registrant_email": "dns@github.com",
        "registrant_phone": "+1.4155550100",
        "registrant_address": "88 Colin P Kelly Jr St, San Francisco, CA 94107, US",
        "status": ["clientDeleteProhibited", "clientTransferProhibited", "clientUpdateProhibited", "serverDeleteProhibited", "serverTransferProhibited", "serverUpdateProhibited"],
        "dnssec": "Signed",
        "dnssec_algorithm": "13 (ECDSAP256SHA256)",
        "privacy_protected": False
    },
    "google.com": {
        "registrar": "MarkMonitor Inc.",
        "registrar_iana_id": 292,
        "registrar_url": "https://www.markmonitor.com",
        "creation_date": "1997-09-15 04:00:00",
        "expiration_date": "2028-09-14 04:00:00",
        "updated_date": "2024-08-14 14:22:00",
        "name_servers": ["ns1.google.com", "ns2.google.com", "ns3.google.com", "ns4.google.com"],
        "registrant_name": "Google LLC",
        "registrant_org": "Google LLC",
        "registrant_email": "dns-admin@google.com",
        "registrant_phone": "+1.6502530000",
        "registrant_address": "1600 Amphitheatre Parkway, Mountain View, CA 94043, US",
        "status": ["clientDeleteProhibited", "clientTransferProhibited", "clientUpdateProhibited", "serverDeleteProhibited", "serverTransferProhibited", "serverUpdateProhibited"],
        "dnssec": "Signed",
        "dnssec_algorithm": "13 (ECDSAP256SHA256)",
        "privacy_protected": False
    }
}

def query_whois(domain):
    """Simulate WHOIS query"""
    domain = domain.lower()
    if domain in SIMULATED_WHOIS:
        return SIMULATED_WHOIS[domain]
    
    # Generate generic response for unknown domains
    return {
        "registrar": "Unknown Registrar",
        "registrar_iana_id": 0,
        "registrar_url": "",
        "creation_date": "Unknown",
        "expiration_date": "Unknown",
        "updated_date": "Unknown",
        "name_servers": [],
        "registrant_name": "Unknown",
        "registrant_org": "Unknown",
        "registrant_email": "Unknown",
        "registrant_phone": "Unknown",
        "registrant_address": "Unknown",
        "status": [],
        "dnssec": "Unknown",
        "privacy_protected": False
    }

def format_whois_report(domain, data):
    """Format WHOIS data into readable report"""
    lines = []
    lines.append(f"{'='*60}")
    lines.append(f"WHOIS LOOKUP REPORT")
    lines.append(f"{'='*60}")
    lines.append(f"Domain: {domain}")
    lines.append(f"")
    
    lines.append(f"--- REGISTRAR INFO ---")
    lines.append(f"Registrar: {data['registrar']}")
    lines.append(f"Registrar IANA ID: {data['registrar_iana_id']}")
    lines.append(f"Registrar URL: {data['registrar_url']}")
    lines.append(f"")
    
    lines.append(f"--- REGISTRATION DATES ---")
    lines.append(f"Creation Date: {data['creation_date']}")
    lines.append(f"Expiration Date: {data['expiration_date']}")
    lines.append(f"Updated Date: {data['updated_date']}")
    
    if data['creation_date'] != "Unknown":
        try:
            created = datetime.fromisoformat(data['creation_date'])
            age = (datetime.now() - created).days
            lines.append(f"Age: {age} days ({age/365.25:.1f} years)")
        except:
            pass
    lines.append(f"")
    
    lines.append(f"--- NAME SERVERS ---")
    for ns in data['name_servers']:
        lines.append(ns)
    lines.append(f"")
    
    lines.append(f"--- REGISTRANT INFO ---")
    lines.append(f"Name: {data['registrant_name']}")
    lines.append(f"Organization: {data['registrant_org']}")
    lines.append(f"Email: {data['registrant_email']}")
    lines.append(f"Phone: {data['registrant_phone']}")
    lines.append(f"Address: {data['registrant_address']}")
    lines.append(f"")
    
    lines.append(f"--- DOMAIN STATUS ---")
    for s in data['status']:
        lines.append(s)
    lines.append(f"")
    
    lines.append(f"--- DNSSEC ---")
    lines.append(f"Signed: {data['dnssec']}")
    if 'dnssec_algorithm' in data:
        lines.append(f"Algorithm: {data['dnssec_algorithm']}")
    lines.append(f"")
    
    lines.append(f"--- PRIVACY PROTECTION ---")
    if data['privacy_protected']:
        lines.append(f"Privacy Protected: Yes")
        if 'privacy_service' in data:
            lines.append(f"Service: {data['privacy_service']}")
    else:
        lines.append(f"Privacy Protected: No")
    
    return "\n".join(lines)

def main():
    test_domains = [
        "example.com",
        "paypal-security-update.tk",
        "microsoft-login-verify.ml",
        "github.com",
        "google.com"
    ]
    
    for domain in test_domains:
        print(f"Performing WHOIS lookup for {domain}...")
        data = query_whois(domain)
        report = format_whois_report(domain, data)
        print(report)
        print(f"\n")

if __name__ == "__main__":
    main()

Performing WHOIS lookup for example.com...
WHOIS LOOKUP REPORT
Domain: example.com

--- REGISTRAR INFO ---
Registrar: ICANN (Reserved)
Registrar IANA ID: 9999
Registrar URL: https://www.iana.org/domains/example

--- REGISTRATION DATES ---
Creation Date: 1995-08-14 04:00:00
Expiration Date: 2026-08-13 04:00:00
Updated Date: 2024-08-14 07:01:00
Age: 11329 days (31.0 years)

--- NAME SERVERS ---
a.iana-servers.net
b.iana-servers.net

--- REGISTRANT INFO ---
Name: ICANN
Organization: Internet Corporation for Assigned Names and Numbers
Email: abuse@iana.org
Phone: +1.3103015800
Address: 12025 Waterfront Drive, Suite 300, Los Angeles, CA 90094, US

--- DOMAIN STATUS ---
clientDeleteProhibited
clientTransferProhibited
clientUpdateProhibited

--- DNSSEC ---
Signed: Signed
Algorithm: 8 (RSASHA256)

--- PRIVACY PROTECTION ---
Privacy Protected: No


Performing WHOIS lookup for paypal-security-update.tk...
WHOIS LOOKUP REPORT
Domain: paypal-security-update.tk

--- REGISTRAR INFO ---
Registrar: Fr

## **Result**
This the program successfully performs WHOIS information collection for a domain as an OSINT investigation technique.